<h2 align="center">Statistical Analysis</h2>
<h5 align="center">This notebook investigates the statistical relationships between meteorological parameters and particulate matter concentrations (PM10 and PM2.5). The objective is to quantify the strength and significance of these relationships using correlation analysis, multiple linear regression, and hypothesis testing. The findings from this notebook will provide statistical evidence before developing predictive machine learning models.</h5>

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import (
    pearsonr,
    spearmanr,
    shapiro,
    levene,
    f_oneway,
    kruskal
)
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [4]:
df = pd.read_excel("../data/processed/mine_data_cleaned.xlsx")
df.columns = df.columns.str.strip()
df.head()

,YEAR,SEASON,MONTH,LOCATION,WEEK,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,PM10,PM2.5
0,2023,WINTER,DECEMBER,HAUL ROAD,1,30.1,64.5,1.0,2.55,56.8,42.6
1,2023,WINTER,DECEMBER,HAUL ROAD,1,29.2,70.3,1.0,2.55,62.4,41.9
2,2023,WINTER,DECEMBER,HAUL ROAD,2,27.6,68.0,0.0,2.55,63.9,43.6
3,2023,WINTER,DECEMBER,HAUL ROAD,2,24.5,85.3,0.0,2.55,65.7,41.9
4,2024,WINTER,DECEMBER,HAUL ROAD,3,28.8,58.9,0.0,2.55,60.2,39.5


In [5]:
for column in [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed",
    "PM10",
    "PM2.5"
]:

    stat, p = shapiro(df[column].sample(500, random_state=42))

    print(column)

    print(f"P-value : {p:.4f}")

    print()

Temperature
P-value : 0.0001

Relative_humidity
P-value : 0.0000

RF-CUMULATIVE
P-value : 0.0000

Wind_speed
P-value : 0.0000

PM10
P-value : 0.0000

PM2.5
P-value : 0.0070



In [6]:
weather = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

for feature in weather:

    r, p = pearsonr(df[feature], df["PM10"])

    print(f"{feature}")

    print(f"Correlation : {r:.3f}")

    print(f"P-value : {p:.4f}")

    print()

Temperature
Correlation : 0.168
P-value : 0.0000

Relative_humidity
Correlation : -0.155
P-value : 0.0000

RF-CUMULATIVE
Correlation : -0.306
P-value : 0.0000

Wind_speed
Correlation : -0.244
P-value : 0.0000



In [7]:
weather = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

for feature in weather:

    r, p = spearmanr(df[feature], df["PM10"])

    print(f"{feature}")

    print(f"Correlation : {r:.3f}")

    print(f"P-value : {p:.4f}")

    print()

Temperature
Correlation : 0.154
P-value : 0.0000

Relative_humidity
Correlation : -0.143
P-value : 0.0000

RF-CUMULATIVE
Correlation : -0.333
P-value : 0.0000

Wind_speed
Correlation : -0.247
P-value : 0.0000



In [11]:
X = df[
    [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed"
    ]
]
y = df["PM10"]
X = sm.add_constant(X)

In [12]:
model = sm.OLS(y, X).fit()

In [13]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   PM10   R-squared:                       0.123
Model:                            OLS   Adj. R-squared:                  0.122
Method:                 Least Squares   F-statistic:                     151.4
Date:                Mon, 29 Jun 2026   Prob (F-statistic):          2.40e-121
Time:                        16:20:17   Log-Likelihood:                -16706.
No. Observations:                4320   AIC:                         3.342e+04
Df Residuals:                    4315   BIC:                         3.345e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                42.6548      2.96

In [21]:
X1 = df[
    [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed"
    ]
]
X1 = sm.add_constant(X)
y1 = pd.to_numeric(y1, errors="coerce")
valid_idx = y1.notna() & X1.notna().all(axis=1)

In [22]:
model_pm25 = sm.OLS(y1.loc[valid_idx], X1.loc[valid_idx]).fit()
print(model_pm25.summary())

                            OLS Regression Results                            
Dep. Variable:                  PM2.5   R-squared:                       0.077
Model:                            OLS   Adj. R-squared:                  0.076
Method:                 Least Squares   F-statistic:                     90.30
Date:                Mon, 29 Jun 2026   Prob (F-statistic):           8.04e-74
Time:                        16:24:30   Log-Likelihood:                -15664.
No. Observations:                4318   AIC:                         3.134e+04
Df Residuals:                    4313   BIC:                         3.137e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                18.4140      2.33

In [23]:
results = pd.DataFrame({
    "Coefficient": model.params,
    "P-value": model.pvalues,
    "Standard Error": model.bse,
    "t-value": model.tvalues
})

display(results)

,Coefficient,P-value,Standard Error,t-value
const,42.654782,8.165627e-46,2.966935,14.376715
Temperature,0.470263,1.887904e-10,0.073645,6.385571
Relative_humidity,0.044009,3.335521e-03,0.014986,2.936632
RF-CUMULATIVE,-0.067659,2.559502e-53,0.004341,-15.584815
Wind_speed,-0.526599,1.385903e-18,0.059579,-8.838681
